# User Study — Generate 10 Example Cases

Selects a diverse set of 10 LIAR test statements (mix of correct/incorrect predictions, including disagreement cases) and generates the full prediction + SHAP words + Gemini explanation for each.


## 1. Load Model, Data, SHAP, Gemini

In [1]:
import pandas as pd
import numpy as np
import torch
import shap
import time
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

test_df = pd.read_csv("liar_test.csv")
train_df = pd.read_csv("liar_train.csv")
MAX_LENGTH = 64

tokenizer = DistilBertTokenizerFast.from_pretrained("./distilbert_liar_final")
model = DistilBertForSequenceClassification.from_pretrained("./distilbert_liar_final")
model.to(device)
model.eval()

def distilbert_predict_proba(texts, batch_size=8):
    all_probs = []
    texts = list(texts)
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        del inputs, logits
        torch.cuda.empty_cache()
    return np.vstack(all_probs)

probs = distilbert_predict_proba(test_df["text"].astype(str).tolist())
test_df["pred"] = probs.argmax(axis=1)
test_df["confidence"] = probs.max(axis=1)
test_df["correct"] = test_df["pred"] == test_df["label_id"]

print(f"Test accuracy: {test_df['correct'].mean():.4f}")


Using device: cpu


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Test accuracy: 0.6886


In [2]:
import re

STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "is", "are", "was", "were", "be", "been", "being",
    "to", "of", "in", "on", "at", "for", "with", "by", "from", "up", "about", "into", "through",
    "we", "us", "our", "you", "your", "i", "he", "she", "it", "they", "them", "his", "her",
    "its", "their", "this", "that", "these", "those", "s", "t", "re", "ve", "ll", "d", "m",
    "have", "has", "had", "do", "does", "did", "will", "would", "could", "should", "can",
    "not", "no", "so", "as", "if", "than", "then", "there", "here", "which", "who", "whom",
    "what", "when", "where", "why", "how",
}

def is_meaningful_token(token):
    clean = re.sub(r"[^a-zA-Z']", "", token.strip())
    if len(clean) <= 1:
        return False
    if clean.lower() in STOPWORDS:
        return False
    return True

masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(distilbert_predict_proba, masker)

def get_top_shap_tokens(text, class_idx, top_k=5):
    """Returns top-k meaningful (non-punctuation, non-stopword) tokens by SHAP value."""
    shap_values = explainer([text])
    tokens = shap_values.data[0]
    values = shap_values.values[0, :, class_idx]
    pairs = list(zip(tokens, values))

    # Filter to meaningful content words only, then rank by absolute SHAP value
    meaningful_pairs = [(t, v) for t, v in pairs if is_meaningful_token(t)]
    meaningful_pairs.sort(key=lambda x: abs(x[1]), reverse=True)
    return meaningful_pairs[:top_k]

print("SHAP explainer ready (with stopword/punctuation filtering).")


SHAP explainer ready (with stopword/punctuation filtering).


In [3]:
from google import genai
import os

API_KEY = os.environ.get("GEMINI_API_KEY", "AQ.Ab8RN6Kkv38FIWJHo9_STGeuijYlLntPXFYffX62JfmukPtlhw")
client = genai.Client(api_key=API_KEY)
GEMINI_MODEL = "gemini-flash-lite-latest"

def build_explanation_prompt(text, predicted_label, confidence, top_tokens):
    fake_words = [t.strip() for t, v in top_tokens if v > 0]
    real_words = [t.strip() for t, v in top_tokens if v < 0]
    return f"""The model classified this statement as [{predicted_label.upper()}] with {confidence:.0%} confidence.

Statement: "{text}"

The most influential words pushing toward FAKE were: {fake_words if fake_words else "none"}.
The most influential words pushing toward REAL were: {real_words if real_words else "none"}.

Write a 2-sentence explanation for a non-technical reader, referencing the specific influential words above. Do not introduce reasoning that isn't grounded in these words."""

def generate_explanation(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
            return response.text.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(3)
            else:
                return f"(unavailable: {e})"

test_response = client.models.generate_content(model=GEMINI_MODEL, contents="Reply with exactly: connected.")
print(test_response.text)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


connected.


## 2. Select 10 Diverse Example Statements

Mix: some correct predictions, some incorrect, and disagreement cases with the baseline.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english", min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])
baseline_pipeline.fit(train_df["text"], train_df["label_id"])
test_df["baseline_pred"] = baseline_pipeline.predict(test_df["text"].astype(str))
test_df["disagreement"] = test_df["baseline_pred"] != test_df["pred"]

# Selection: 5 correct, 3 incorrect, 2 disagreement cases (adjust as needed)
correct_cases = test_df[test_df["correct"]].sample(5, random_state=42)
incorrect_cases = test_df[~test_df["correct"]].sample(3, random_state=42)
disagreement_cases = test_df[test_df["disagreement"] & ~test_df.index.isin(correct_cases.index) & ~test_df.index.isin(incorrect_cases.index)].sample(2, random_state=42)

selected = pd.concat([correct_cases, incorrect_cases, disagreement_cases]).drop_duplicates().reset_index(drop=True)
print(f"Selected {len(selected)} examples")
selected[["text", "label", "pred", "confidence", "correct", "disagreement"]]


Selected 10 examples


,text,label,pred,confidence,correct,disagreement
0,Scientists tell us that we could have a cure i...,fake,1,0.881224,True,False
1,"We were not, I repeat, were not told that wate...",fake,1,0.649185,True,True
2,Even among second and third generation Muslims...,fake,1,0.550916,True,True
3,A company hired to do Common Core testing in F...,fake,1,0.650147,True,False
4,Sen. Bob Menendez voted to enact a new tax on ...,fake,1,0.565809,True,True
5,"Quite frankly, it was during the Bush years of...",real,1,0.711702,False,True
6,Its been since 1888 that a Senate of a differe...,fake,0,0.600817,False,True
7,Says that his transportation budget and Ohio T...,fake,0,0.500457,False,False
8,Says state Senate President Stephen Sweeney ga...,fake,1,0.732031,True,True
9,Rebuilding three high schools will benefit 40 ...,fake,1,0.662142,True,True


## 3. Generate Full Output for Each Example

Prediction, SHAP words, and Gemini explanation — formatted for copy into the survey.


In [5]:
results = []

for idx, row in selected.iterrows():
    predicted_label = "Fake" if row["pred"] == 1 else "Real"
    top_tokens = get_top_shap_tokens(row["text"], class_idx=int(row["pred"]), top_k=5)
    prompt = build_explanation_prompt(row["text"], predicted_label, row["confidence"], top_tokens)
    explanation = generate_explanation(prompt)

    result = {
        "number": len(results) + 1,
        "text": row["text"],
        "true_label": row["label"],
        "predicted_label": predicted_label,
        "confidence": row["confidence"],
        "correct": bool(row["correct"]),
        "top_tokens": [(t.strip(), float(v)) for t, v in top_tokens],
        "explanation": explanation,
    }
    results.append(result)

    print(f"[{result['number']}/10] Done: {row['text'][:60]}...")
    time.sleep(4.5)

print("\nAll examples generated.")


  0%|          | 0/498 [00:00<?, ?it/s]


PartitionExplainer explainer: 100%|██████████| 1/1 [00:00<?, ?it/s]


PartitionExplainer explainer: 2it [01:06, 66.57s/it]               

[1/10] Done: Scientists tell us that we could have a cure in 10 years for...


  0%|          | 0/498 [00:00<?, ?it/s]

...

  0%|          | 0/240 [00:00<?, ?it/s]


PartitionExplainer explainer: 100%|██████████| 1/1 [00:00<?, ?it/s]


PartitionExplainer explainer: 2it [00:17, 17.09s/it]               

[10/10] Done: Rebuilding three high schools will benefit 40 percent of Por...



All examples generated.


## 4. Formatted Output


In [6]:
for r in results:
    print(f"=== Example {r['number']} ===")
    print(f"Statement: {r['text']}")
    print(f"Prediction: {r['predicted_label']} ({r['confidence']:.0%} confidence)")
    print(f"True label: {r['true_label']} {'[CORRECT]' if r['correct'] else '[INCORRECT]'}")
    print("Top words:")
    for token, val in r["top_tokens"]:
        direction = "toward FAKE" if val > 0 else "toward REAL"
        print(f"  {token}: {val:+.3f} ({direction})")
    print(f"Explanation: {r['explanation']}")
    print()


=== Example 1 ===
Statement: Scientists tell us that we could have a cure in 10 years for Alzheimers were it not for overzealous regulators, excessive taxation and greedy litigators.
Prediction: Fake (88% confidence)
True label: fake [CORRECT]
Top words:
  taxation: +0.034 (toward FAKE)
  excessive: +0.028 (toward FAKE)
  Scientists: +0.018 (toward FAKE)
  greedy: +0.015 (toward FAKE)
  tors: +0.014 (toward FAKE)
Explanation: The model flagged the statement as fake largely due to loaded and emotional words like **"excessive," "taxation," "greedy,"** and **"tors"** (referring to litigators). Additionally, invoking authority through the word **"Scientists"** to blame regulations and taxes pushed the model's confidence toward a classification of fake.

=== Example 2 ===
Statement: We were not, I repeat, were not told that waterboarding or any of these other enhanced interrogation methods were used.
Prediction: Fake (65% confidence)
True label: fake [CORRECT]
Top words:
  told: +0.011 (tow

In [7]:
import json

with open("user_study_examples.json", "w") as f:
    json.dump(results, f, indent=2, default=float)

print("Saved: user_study_examples.json")


Saved: user_study_examples.json
